# 폰트설정
- 아래 코드 선 실행
- seaborn, matplotlib 라이브러리 활용 시, 한글폰트 깨짐 방지
- 런타임 > 세션 다시 시작
  + 아래 코드 생략, 다음 코드 실행
- Google Colab

In [ ]:
# ── 환경 설정(선택) ─────────────────────────────────────────────────────
# Colab·리눅스 등에서 폰트 설치 셀 — 로컬 Windows는 생략 가능

# !sudo apt-get install -y -qq fonts-nanum
# !fc-cache -fv
# !rm ~/.cache/matplotlib -rf


In [ ]:
# ── Matplotlib ────────────────────────────────────────────────────
# 그래프·축·스타일

import platform

import matplotlib.pyplot as plt
from matplotlib import font_manager

def setup_matplotlib_korean_font() -> str | None:
    """Windows / Mac / 기타 순으로 한글 폰트를 try-except로 적용."""
    try:
        plt.rcParams["axes.unicode_minus"] = False
    except Exception:
        pass

    system = platform.system()

    if system == "Windows":
        candidates = ["Malgun Gothic", "맑은 고딕"]
    elif system == "Darwin":
        candidates = ["Apple SD Gothic Neo", "AppleGothic"]
    else:
        candidates = ["NanumGothic", "Noto Sans CJK KR", "Malgun Gothic"]

    for family in candidates:
        try:
            prop = font_manager.FontProperties(family=family)
            path = font_manager.findfont(prop, fallback_to_default=False)
            if path and "dejavu" not in path.lower():
                plt.rcParams["font.family"] = family
                print(f"[한글 폰트] 적용: {family}")
                return family
        except Exception as exc:
            print(f"[한글 폰트] 후보 실패 ({family}): {exc}")
            continue

    # 검증 없이 플랫폼 기본 이름만 지정 (실제 글리프는 런타임에 확인)
    try:
        if system == "Windows":
            plt.rcParams["font.family"] = "Malgun Gothic"
            print("[한글 폰트] 폴백(Windows): Malgun Gothic")
            return "Malgun Gothic"
        if system == "Darwin":
            plt.rcParams["font.family"] = "Apple SD Gothic Neo"
            print("[한글 폰트] 폴백(macOS): Apple SD Gothic Neo")
            return "Apple SD Gothic Neo"
    except Exception as exc:
        print(f"[한글 폰트] 폴백 실패: {exc}")

    print("[한글 폰트] 시스템 기본만 사용 — 한글이 깨질 수 있습니다.")
    return None

setup_matplotlib_korean_font()


# Part 2. 정형 데이터 Pandas 튜토리얼 70제 (`dataset/정형`)

`Pandas예제1`~`7`의 학습 흐름(불러오기, `info`/`describe`, `loc`/`iloc`, 조건 필터, 정렬, `groupby`, 결측·중복, 파생, 문자열, 날짜, `merge`, NumPy)을 **아래 세 CSV**에 적용합니다.

| 데이터 | 파일(예시 이름) |
|--------|-----------------|
| 서버 점검 | `서버유지보수_보정데이터(0512).csv` |
| 군수품 청구 | `군수품자동청구_보정데이터(0512).csv` |
| 장병 복지시설 | `장병복지시설용_보완데이터(0512).csv` |

- 경로: 노트북을 `소집교육/0423`에서 열면 `../dataset/정형/`, 프로젝트 루트에서 열면 `소집교육/dataset/정형/`을 자동 탐색합니다.
- 파일명이 유니코드(NFC/NFD)로 달라도 **컬럼 시그니처**(`log_id`, `supply_id`, `soldier_id`)로 구분합니다.

**문제 구성**: `#` 큰 주제 → `##` 세부 주제 → `### 문제 N.` (목표·설명·실습·힌트) → 코드 셀.



## 0. 정형 데이터 로드

**아래 셀을 Part 2 시작 전에 한 번 실행**하세요. `df_srv`, `df_sup`, `df_wel`이 생성됩니다.


In [ ]:
# 코드

# 1. 데이터 읽기와 구조 이해



## 1.1 크기·타입·스키마



### 문제 1.
- **목표**: 데이터프레임의 행·열 규모를 파악한다.
- **설명**: `shape` 속성은 `(행 개수, 열 개수)` 형태의 튜플을 반환합니다. 대용량 여부, 다른 테이블과 행 수 비교, 메모리 추정의 출발점이 됩니다.
- **실습**: `df_srv`의 행·열 개수(`shape`)를 출력하세요.
- **힌트**: `df_srv.shape[0]`은 행, `shape[1]`은 열입니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 2.
- **목표**: 열 이름, dtypes, 비결측 개수 등 스키마를 한 번에 점검한다.
- **설명**: `info()`는 각 열의 이름·dtype·비어 있지 않은 값 개수를 보여 줍니다. `object`는 주로 문자열이며, 날짜가 문자열로 들어온 경우 나중에 `to_datetime`이 필요할 수 있습니다.
- **실습**: `df_sup`에 대해 `info()`를 호출해 dtypes와 결측을 확인하세요.
- **힌트**: 맨 아래 `memory usage`로 대략적인 메모리도 확인할 수 있습니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 3.
- **목표**: 수치형 열의 분포(평균, 사분위, 최소·최대)를 요약한다.
- **설명**: `describe()`는 기본적으로 수치형 열만 대상으로 합니다. 복지 데이터의 만족도·거리 등 스케일이 다른 열을 비교할 때 유용합니다.
- **실습**: `df_wel`의 수치형 열만 `describe()`로 요약하세요.
- **힌트**: 범주형 요약은 `describe(include='object')`로 별도 확인할 수 있습니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 4.
- **목표**: 분석·코딩 시 참조할 열 이름 목록을 얻는다.
- **설명**: `columns`는 `Index` 객체이며 `.tolist()`로 파이썬 리스트로 바꿀 수 있습니다. 오타 방지와 동적 컬럼 선택에 쓰입니다.
- **실습**: `df_srv.columns.tolist()`로 컬럼 이름 리스트를 보세요.
- **힌트**: 첫 열은 `df.columns[0]`처럼 인덱싱할 수 있습니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 5.
- **목표**: 열 단위로 dtype을 확인한다.
- **설명**: `dtypes`는 Series로, 열 이름 → dtype 매핑입니다. 병합·연산 전에 `int`와 `object` 혼동을 줄입니다.
- **실습**: `df_sup.dtypes`를 출력하세요.
- **힌트**: `df['col'].dtype`으로 한 열만 확인할 수도 있습니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



## 1.2 미리보기·표본·메모리



### 문제 6.
- **목표**: 앞쪽 행 몇 개로 값의 형태와 단위를 육안 확인한다.
- **설명**: `head(n)`은 상위 n행입니다. 실제 업무에서는 샘플과 함께 도메인(부대 코드, 품목명 등)이 기대와 맞는지 봅니다.
- **실습**: `df_wel.head(7)`로 앞 7행을 보세요.
- **힌트**: 기본값은 `head()`만 쓰면 5행입니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 7.
- **목표**: 데이터 끝부분(최근 적재 행 등)을 확인한다.
- **설명**: `tail(n)`은 하위 n행입니다. 시계열이 정렬되어 있다면 최근 기록이 아래에 모일 수 있습니다.
- **실습**: `df_srv.tail(3)`로 끝 3행을 보세요.
- **힌트**: 정렬 없이 tail만으로 ‘최근’을 단정하긴 어렵습니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 8.
- **목표**: 데이터프레임이 차지하는 메모리를 대략 추정한다.
- **설명**: `memory_usage(deep=True)`는 객체 열의 내용까지 더 정밀하게 잡습니다. 합치면 전체 규모의 감을 잡을 수 있습니다.
- **실습**: `df_sup.memory_usage(deep=True).sum()`으로 대략적 메모리를 구하세요.
- **힌트**: 단위는 바이트이므로 1e6으로 나누면 MB 근사가 됩니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 9.
- **목표**: 무작위 표본으로 편향된 앞부분만 보는 것을 완화한다.
- **설명**: `sample(n, random_state=)`는 재현 가능한 난수 표본입니다. EDA에서 무작위 몇 행을 보는 습관이 좋습니다.
- **실습**: `df_wel.sample(5, random_state=42)`로 무작위 5행을 보세요.
- **힌트**: `random_state`를 고정해야 노트북 결과가 매번 같습니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 10.
- **목표**: 세 테이블의 행 수를 빠르게 비교한다.
- **설명**: `len(df)`는 행 수와 동일합니다. `shape[0]`과 같습니다. 로그(서버)·거래(군수품)·개인 단위(복지) 규모 차이를 감 잡습니다.
- **실습**: 세 데이터프레임 각각 `len(df)`로 행 수를 출력하세요.
- **힌트**: 열 수는 `len(df.columns)`입니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



# 2. 행·열 선택



## 2.1 열 이름 기반 (`loc`, `[]`)



### 문제 11.
- **목표**: `loc`으로 열 이름 기반 슬라이싱을 연습한다.
- **설명**: `loc[행조건, 열리스트]`에서 `:`는 모든 행입니다. 가독성이 좋아 실무에서 자주 씁니다.
- **실습**: `df_srv.loc[:, ['server_id', 'location', 'cpu_usage']].head()`
- **힌트**: 행을 먼저 필터한 뒤 같은 열만 쓰면 분석 파이프라인이 명확해집니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 12.
- **목표**: 대괄호 `[]`로 열 부분집합을 만든다.
- **설명**: 단일 열은 Series, 열 리스트는 DataFrame을 반환합니다.
- **실습**: `df_sup[['item_name', 'category', 'stock_level']].head()`
- **힌트**: 한 열만 쓸 때는 `df['col']` 이중 대괄호가 아닙니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 13.
- **목표**: 복지 데이터에서 식별자·지역·만족도 등 핵심 열만 추린다.
- **설명**: 민감·개인 데이터 다룰 때는 필요한 열만 선택하는 습관이 중요합니다.
- **실습**: `df_wel.loc[:, ['soldier_id', 'region', 'satisfaction_score']].head()`
- **힌트**: 나중에 `merge`할 키 열을 항상 포함했는지 확인하세요.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



## 2.2 위치 기반 (`iloc`)



### 문제 14.
- **목표**: `iloc` 정수 위치 인덱싱으로 첫 행을 가져온다.
- **설명**: `iloc[0]`은 첫 번째 행(Series)입니다. 라벨 인덱스와 무관하게 ‘위치’로 접근합니다.
- **실습**: `df_srv.iloc[0, :]` 첫 행 전체
- **힌트**: 마지막 행은 `iloc[-1]`입니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 15.
- **목표**: 행·열 범위를 슬라이싱한다.
- **설명**: `iloc[a:b, c:d]`는 파이썬 슬라이스 규칙(끝 미포함)을 따릅니다.
- **실습**: `df_sup.iloc[:5, :3]` 앞 5행·앞 3열
- **힌트**: `iloc[:, -1]`은 마지막 열 전체입니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 16.
- **목표**: 컬럼명을 코드로 읽어 동적 선택한다.
- **설명**: 첫 열이 항상 ID라는 가정이 맞는지 데이터 정의서와 맞춰야 합니다.
- **실습**: 첫 번째 컬럼명을 변수에 담아 `df_wel.loc[:, [첫컬럼]].head()`
- **힌트**: `df.columns[n]`으로 n번째 열 이름을 쓸 수 있습니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 17.
- **목표**: 인덱스의 일부와 특정 열만 조합한다.
- **설명**: `df.index[:10]`은 앞 10개 인덱스 라벨입니다. 정수 기본 인덱스면 0~9와 대응됩니다.
- **실습**: `df_srv.loc[df_srv.index[:10], ['log_id', 'check_type']]`
- **힌트**: 필터 후 `reset_index`하면 인덱스가 바뀔 수 있습니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 18.
- **목표**: 중간 구간 행을 `iloc`으로 집어 본다.
- **설명**: 대용량에서 중간 행을 보면 앞·뒤와 다른 패턴(결측, 이상값)이 있는지 힌트가 됩니다.
- **실습**: `df_sup.iloc[100:105]` 100~104행 전체 열
- **힌트**: 범위를 바꿔 여러 구간을 샘플링해 보세요.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



# 3. 조건 필터링



## 3.1 단일 조건



### 문제 19.
- **목표**: 단일 조건으로 행을 필터링한다.
- **설명**: 불리언 마스크 `df['col'] == 값`을 `loc`의 첫 인자에 넣습니다.
- **실습**: `issue_category`가 `'디스크'`인 서버 점검 행만 `loc`으로.
- **힌트**: 문자열은 대소문자·공백까지 정확히 일치해야 합니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 20.
- **목표**: 군수품 범주(`category`)로 서브셋을 만든다.
- **설명**: 품목군별로 재고 정책이 다를 수 있어 범주 필터는 자주 씁니다.
- **실습**: `category`가 `'의료'`인 군수품 행만.
- **힌트**: 고유 범주는 `df['category'].unique()`로 확인할 수 있습니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 21.
- **목표**: 최빈값(또는 특정) 범주로 필터링한다.
- **설명**: `value_counts().index[0]`은 가장 많이 나온 `region`입니다. 지역 편중 여부를 볼 때 유용합니다.
- **실습**: `region`에서 가장 빈도가 높은 값 하나를 골라, 해당 지역 행만 필터링.
- **힌트**: 특정 지역명을 알고 있다면 `== '지역명'`으로 바꿔도 됩니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



## 3.2 복합 조건·`isin`·비교



### 문제 22.
- **목표**: AND(`&`)로 두 조건을 동시에 만족하는 행만 남긴다.
- **설명**: 각 조건은 괄호로 감싸야 합니다. `and` 대신 `&`를 씁니다.
- **실습**: `fix_required == 1` 이고 `issue_detected == 1` 인 행 (AND).
- **힌트**: OR는 `|` 입니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 23.
- **목표**: 범주 값이 여러 개 중 하나일 때 `isin`을 쓴다.
- **설명**: 긴 OR 체인 대신 리스트로 깔끔하게 씁니다.
- **실습**: `check_type`이 `'일간'` 또는 `'주간'` (`isin`).
- **힌트**: 부정은 `~df['col'].isin([...])` 입니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 24.
- **목표**: 이슈 유형 복수 선택으로 CPU·디스크 관련 로그만 본다.
- **설명**: 운영 이슈 유형별 SLA를 볼 때 자주 하는 패턴입니다.
- **실습**: `issue_category`가 `['CPU과부하', '디스크']` 중 하나 (`isin`).
- **힌트**: 카테고리 순서는 리스트에만 영향 없고 필터 결과는 동일합니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 25.
- **목표**: 수치 임계값으로 부하 높은 서버만 본다.
- **설명**: 임계값은 도메인 합의(예: 40%)가 필요합니다.
- **실습**: `cpu_usage`가 40 초과인 행.
- **힌트**: 경계 포함은 `>=` 로 조정합니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 26.
- **목표**: 두 열의 행별 비교로 ‘재주문 필요’ 상황에 가까운 행을 찾는다.
- **설명**: 실제 비즈니스 룰과 다를 수 있으나 EDA 패턴으로 유용합니다.
- **실습**: `stock_level`이 `reorder_threshold` 미만인 행.
- **힌트**: 같은 행의 두 열을 비교할 때는 각각 Series이므로 브로드캐스트됩니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 27.
- **목표**: 정신 건강 점수 구간으로 서브셋을 잡는다.
- **설명**: 구간 경계는 보고 목적에 맞게 조정합니다.
- **실습**: `mental_wellbeing_score`가 60 이상 80 이하.
- **힌트**: `between(60, 80)` 메서드로도 같은 조건을 쓸 수 있습니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 28.
- **목표**: 필터 후 인덱스를 0부터 다시 매긴다.
- **설명**: 후속 `concat`·저장 시 깔끔한 인덱스가 필요할 때 씁니다.
- **실습**: 필터 후 `reset_index(drop=True)`로 인덱스 초기화.
- **힌트**: `drop=False`면 이전 인덱스가 열로 남습니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



# 4. 정렬



## 4.1 `sort_values`



### 문제 29.
- **목표**: `sort_values`로 CPU 사용률 상위 서버를 본다.
- **설명**: 상위 n개는 `head`와 조합합니다.
- **실습**: `df_srv`를 `cpu_usage` 내림차순 정렬 후 상위 8행.
- **힌트**: `na_position`으로 결측 위치를 제어할 수 있습니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 30.
- **목표**: 청구 수량이 적은 순으로 정렬한다.
- **설명**: 오름차순이 기본(`ascending=True`)입니다.
- **실습**: `df_sup`를 `requested_quantity` 오름차순.
- **힌트**: 복수 열 정렬은 리스트로 열 이름을 넘깁니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 31.
- **목표**: 시설까지 거리가 먼 순으로 복지 레코드를 본다.
- **설명**: 접근성 불균형 분석의 출발점입니다.
- **실습**: `df_wel`를 `distance_to_facility` 내림차순.
- **힌트**: 지리 좌표(`lat`,`lon`)와 함께 보면 지도 시각화로 이어질 수 있습니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 32.
- **목표**: 다중 키 정렬로 같은 부대 내 시간 순서 등을 맞춘다.
- **설명**: 먼저 쓴 열이 1차 정렬 키입니다.
- **실습**: `location`, `check_date` 기준 다중 정렬 (오름차순).
- **힌트**: `ascending=[True, False]`처럼 열마다 다르게 줄 수 있습니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



# 5. 그룹 집계



## 5.1 `groupby`·`agg`



### 문제 33.
- **목표**: `groupby`로 부대(`location`)별 점검 건수를 센다.
- **설명**: 집계 함수로 `count`,`size` 등을 씁니다. `count`는 비결측만 셉니다.
- **실습**: `location`별 행 수 (`log_id` count).
- **힌트**: `size()`는 그룹 크기(행 수)로 결측과 무관합니다.


In [ ]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환



### 문제 34.
- **목표**: 이슈 유형별 평균 CPU를 비교한다.
- **설명**: 그룹별 대표값 비교는 운영 인사이트의 기본입니다.
- **실습**: `issue_category`별 `cpu_usage` 평균.
- **힌트**: 같은 패턴으로 `median`,`std`도 가능합니다.


In [ ]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환



### 문제 35.
- **목표**: `agg`로 그룹당 여러 통계를 한 번에 구한다.
- **설명**: 점검 주기별로 수리 시간 평균·최대를 동시에 봅니다.
- **실습**: `check_type`별 `fix_duration_hours` 평균·최대 (`agg`).
- **힌트**: 딕셔너리로 열마다 다른 함수를 줄 수도 있습니다.


In [ ]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환



### 문제 36.
- **목표**: 품목 범주별 청구량 합계와 평균을 동시에 본다.
- **설명**: 소계·평균을 같이 보면 ‘총량은 크지만 건당 작다’ 같은 패턴이 보입니다.
- **실습**: `category`별 `requested_quantity` 합·평균.
- **힌트**: `groupby` 후 `transform`이면 원본 행에 붙이기도 합니다.


In [ ]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환



### 문제 37.
- **목표**: 부대 코드별 평균 재고 수준 상위를 본다.
- **설명**: 자동발주·물류 우선순위 논의에 쓸 수 있습니다.
- **실습**: `unit_code`별 `stock_level` 평균 상위 10개 부대.
- **힌트**: 건수가 적은 부대는 평균이 왜곡될 수 있어 `count`와 함께 보세요.


In [ ]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환



### 문제 38.
- **목표**: 시설 유형별 만족도 평균을 비교한다.
- **설명**: 정책별(체육관·독서실 등) 만족도 차이를 가설 검정 전에 봅니다.
- **실습**: `facility_type`별 `satisfaction_score` 평균.
- **힌트**: 표본 수가 다르면 신뢰구간·검정이 필요합니다.


In [ ]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환



### 문제 39.
- **목표**: 계급(`rank`)별 정신건강 점수 중앙값을 본다.
- **설명**: 이상치에 덜 민감한 대표값입니다.
- **실습**: `rank`별 `mental_wellbeing_score` 중앙값.
- **힌트**: `mean`과 `median`을 나란히 비교하면 분포 꼬리를 짐작합니다.


In [ ]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환



### 문제 40.
- **목표**: 지역×시설 유형 교차로 만족도를 본다.
- **설명**: 다차원 그룹은 `groupby([col1,col2])` 입니다.
- **실습**: `region`·`facility_type` 다중 그룹 평균 만족도.
- **힌트**: 결과가 길면 `unstack`으로 피벗 형태를 만들 수 있습니다.


In [ ]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환



### 문제 41.
- **목표**: `groupby().size()`로 플래그별 건수를 센다.
- **설명**: `value_counts`와 유사하지만 그룹 객체 체인에 맞출 때 씁니다.
- **실습**: `auto_reorder_flag`별 건수 (`groupby` size).
- **힌트**: `as_index=False`로 표 형태 DataFrame을 만들 수 있습니다.


In [ ]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환



### 문제 42.
- **목표**: 이슈 탐지 여부별로 수리 필요 플래그 평균을 본다.
- **설명**: 라벨 간 연관(간단한 비율)을 탐색합니다.
- **실습**: `issue_detected`별 `fix_required` 평균.
- **힌트**: 0~1 평균은 비율 해석이 가능합니다.


In [ ]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환



# 6. 데이터 품질과 파생



## 6.1 결측·중복



### 문제 43.
- **목표**: 열 단위 결측 개수로 데이터 품질을 점검한다.
- **설명**: `isna().sum()`은 열마다 결측 수입니다.
- **실습**: `df_srv.isna().sum()` 결측 개수 열별.
- **힌트**: 전체 결측은 `df.isna().sum().sum()` 입니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 44.
- **목표**: 수치 열 결측을 중앙값으로 채워 단순 대체를 연습한다.
- **설명**: 실무에서는 KNN·모델 기반 대체 등 더 정교한 방법도 고려합니다.
- **실습**: `df_wel` 숫자 열 결측을 열 중앙값으로 채운 뒤 남은 결측 총합.
- **힌트**: 범주형 결측은 `fillna('Unknown')` 등 별도 처리가 필요합니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 45.
- **목표**: 완전 중복 행 개수를 센다.
- **설명**: 모든 열이 동일한 행이 몇 개인지 봅니다.
- **실습**: `df_sup.duplicated().sum()` 전체 중복 행 개수.
- **힌트**: 특정 열 기준은 `subset=[...]` 입니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 46.
- **목표**: 기본 키 후보(`supply_id`)로 중복을 제거한 유일 행 수를 본다.
- **설명**: 실제 PK 정의와 일치하는지 검증하는 단계입니다.
- **실습**: `supply_id` 기준 중복 제거 후 행 수.
- **힌트**: `keep='last'` 등으로 어떤 행을 남길지 정할 수 있습니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



## 6.2 파생 변수·구간화



### 문제 47.
- **목표**: 여러 부하 지표를 더한 파생 열을 만든다.
- **설명**: `assign`은 새 DataFrame을 반환하므로 원본 보존에 유리합니다.
- **실습**: 파생: `usage_sum = cpu_usage + memory_usage + disk_usage` (`assign`).
- **힌트**: 가중합이 필요하면 `w1*cpu + w2*mem` 형태로 확장합니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 48.
- **목표**: 연속 비율을 구간 범주로 나눈다.
- **설명**: `pd.cut`은 구간 경계와 라벨을 정의합니다.
- **실습**: `weekend_usage_ratio`를 `pd.cut`으로 low/mid/high 구간 빈도.
- **힌트**: 구간을 `qcut`으로 분위수 기준으로 나눌 수도 있습니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



# 7. 문자열 처리



## 7.1 `str` 접근자



### 문제 49.
- **목표**: 품목명 문자열을 대문자로 통일한다.
- **설명**: JOIN·중복 제거 전 정규화에 쓰입니다.
- **실습**: `df_sup['item_name'].str.upper().head()`
- **힌트**: `str.lower`, `str.title`도 같은 계열입니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 50.
- **목표**: `str.contains`로 문자열 패턴 포함 여부를 센다.
- **설명**: `na=False`로 결측은 False 처리해 연산 오류를 막습니다.
- **실습**: `df_srv['location'].str.contains('사단', na=False)` True 개수.
- **힌트**: 정규식은 `regex=True`(기본)와 패턴 문자열을 사용합니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 51.
- **목표**: 부대 코드에서 앞 두 글자만 잘라 상위 체계를 만든다.
- **설명**: 문자열 Series에 `str` 접근자를 씁니다.
- **실습**: `df_wel['unit_code']`를 문자열로 바꾼 뒤 앞 두 글자.
- **힌트**: 숫자형이면 `astype(str)`이 안전합니다.


In [ ]:
# ── dtype 변환 ──────────────────────────────────────────────────────
# `astype`으로 자료형 변경



### 문제 52.
- **목표**: 범주 이름 길이의 평균으로 명명 패턴을 대략 본다.
- **설명**: `str.len()`은 문자 수입니다.
- **실습**: `category` 문자열 길이 `str.len()` 평균.
- **힌트**: 바이트 길이가 아닌 유니코드 문자 기준입니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



# 8. 시계열 다루기



## 8.1 변환·추출·필터·집계



### 문제 53.
- **목표**: 문자열로 된 점검 시각을 datetime으로 바꾼다.
- **설명**: 시계열 연산·정렬·리샘플의 전제입니다.
- **실습**: `check_date`를 `pd.to_datetime` 변환 후 dtype 확인.
- **힌트**: `errors='coerce'`로 깨진 값은 NaT가 됩니다.


In [ ]:
# ── 날짜·시간 파싱 ──────────────────────────────────────────────────────
# `to_datetime` 등 시계열 전처리



### 문제 54.
- **목표**: 점검 연도별 건수로 연도 편중을 본다.
- **설명**: `.dt` 접근자는 datetime Series에만 사용합니다.
- **실습**: 점검일에서 연도 `.dt.year` 빈도.
- **힌트**: 월별은 `dt.month`, 분기는 `dt.quarter`입니다.


In [ ]:
# ── 날짜·시간 파싱 ──────────────────────────────────────────────────────
# `to_datetime` 등 시계열 전처리



### 문제 55.
- **목표**: 청구일을 월 단위로 집계한다.
- **설명**: `to_period('M')`은 월 구간 라벨을 만듭니다.
- **실습**: `request_date` datetime 변환 후 월별 건수.
- **힌트**: `resample`은 인덱스가 DatetimeIndex일 때 특히 편합니다.


In [ ]:
# ── 날짜·시간 파싱 ──────────────────────────────────────────────────────
# `to_datetime` 등 시계열 전처리



### 문제 56.
- **목표**: 특정 일 이후의 서버 점검만 남긴다.
- **설명**: 문자열과 비교해도 pandas가 datetime으로 맞춰 줄 때가 많습니다.
- **실습**: 2024-10-01 이후 서버 점검만 필터.
- **힌트**: 구간은 `between`으로도 표현할 수 있습니다.


In [ ]:
# ── 날짜·시간 파싱 ──────────────────────────────────────────────────────
# `to_datetime` 등 시계열 전처리



### 문제 57.
- **목표**: 기록 일자의 요일 분포를 본다.
- **설명**: `dayofweek`는 월=0 … 일=6 (pandas 기본).
- **실습**: `df_wel['date']`의 요일(`dayofweek`) 빈도.
- **힌트**: 요일 이름 매핑은 `map`으로 붙일 수 있습니다.


In [ ]:
# ── 날짜·시간 파싱 ──────────────────────────────────────────────────────
# `to_datetime` 등 시계열 전처리



### 문제 58.
- **목표**: 월별 이슈 탐지 건수 합으로 추세를 본다.
- **설명**: 파생 월 열을 만든 뒤 `groupby`합니다.
- **실습**: 월별 `issue_detected` 합.
- **힌트**: 결측 날짜 행은 제외되거나 NaT 그룹이 생길 수 있습니다.


In [ ]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환



### 문제 59.
- **목표**: 청구일 순으로 데이터를 정렬한다.
- **설명**: datetime 열을 만든 뒤 `sort_values`합니다.
- **실습**: `request_date` 기준 정렬.
- **힌트**: 시간대가 섞이면 `tz_localize` 등이 필요할 수 있습니다.


In [ ]:
# ── 날짜·시간 파싱 ──────────────────────────────────────────────────────
# `to_datetime` 등 시계열 전처리



### 문제 60.
- **목표**: 점검 기간의 최소·최대 날짜로 데이터 범위를 확인한다.
- **설명**: 리포트의 ‘분석 기간’ 문구에 씁니다.
- **실습**: `check_date` 최소·최대.
- **힌트**: `idxmin`으로 최소 날짜가 있는 행 인덱스도 찾을 수 있습니다.


In [ ]:
# ── 날짜·시간 파싱 ──────────────────────────────────────────────────────
# `to_datetime` 등 시계열 전처리



# 9. 테이블 결합·변형



## 9.1 `merge`·`concat`·`pivot_table`



### 문제 61.
- **목표**: 군수품과 복지 데이터를 `unit_code`로 내부 조인한다.
- **설명**: inner는 양쪽 모두 키가 있는 행만 남깁니다. 행 수 폭증에 주의합니다.
- **실습**: `df_sup`와 `df_wel`을 `unit_code`로 inner merge 후 `shape`.
- **힌트**: `suffixes`로 겹치는 열 이름을 구분합니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 62.
- **목표**: 조인 결과에서 품목 범주×지역 교차 빈도를 본다.
- **설명**: 다대다 조인이면 행이 늘어난 ‘중복 의미’를 이해해야 합니다.
- **실습**: `unit_code`로 merge한 뒤 `category`·`region`별 행 수 (단일 셀에서 완료).
- **힌트**: 필요하면 `drop_duplicates`로 단위를 맞춥니다.


In [ ]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환



### 문제 63.
- **목표**: 집계 테이블을 원본에 붙여 위치별 평균 CPU를 각 행에 표시한다.
- **설명**: SQL의 윈도 없이 ‘부대 평균’을 열로 붙이는 패턴입니다.
- **실습**: `location`별 평균 `cpu_usage`를 구해 원본 `df_srv`에 `merge`로 붙이기.
- **힌트**: 같은 `location`이면 같은 평균이 반복됩니다.


In [ ]:
# ── groupby 집계 ────────────────────────────────────────────────────
# 범주별 요약·변환



### 문제 64.
- **목표**: `concat`으로 행 방향으로 이어 붙인다.
- **설명**: axis=0이 세로 결합입니다. 인덱스가 중복될 수 있습니다.
- **실습**: `pd.concat`으로 `df_srv`의 0~1행과 2~3행을 세로 결합.
- **힌트**: `ignore_index=True`로 인덱스를 새로 매길 수 있습니다.


In [ ]:
# ── Pandas 실습 ─────────────────────────────────────────────────────



### 문제 65.
- **목표**: `pivot_table`로 범주별 대표값을 표 형태로 만든다.
- **설명**: 엑셀 피벗과 대응됩니다.
- **실습**: `category`별 평균 `avg_daily_usage` (`pivot_table`).
- **힌트**: `columns` 인자로 넓은 표를 만들 수 있습니다.


In [ ]:
# ── 피벗·교차표 ────────────────────────────────────────────────────────
# 요약 표 재배열

